In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_interaction_whl(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "InteractionId	string,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "InteractionKey	string,"
        #     "InteractionCode	int,"
        #     "ExposureKey	string,"
        #     "PartyKey	int,"
        #     "CDI	string,"
        #     "InteractionTypeCode	string,"
        #     "InteractionTypeKey	string,"
        #     "InteractionType	string,"
        #     "InteractionGroupTypeCode	int,"
        #     "InteractionGroupType	string,"
        #     "InteractionCategoryCode	int,"
        #     "InteractionCategory	string,"
        #     "DialerTypeCode	string,"
        #     "DialerType	string,"
        #     "ContactTypeCode	string,"
        #     "ContactType	string,"
        #     "AttemptTypeCode	string,"
        #     "AttemptType	string,"
        #     "AttemptOutcomeCode	string,"
        #     "AttemptOutcome	string,"
        #     "AttemptSpecialMetricCode	string,"
        #     "AttemptSpecialMetric	string,"
        #     "IsPositiveFlag	boolean,"
        #     "IsPromiseFlag	boolean,"
        #     "IsActiveFlag	boolean,"
        #     "IsLegalFlag	boolean,"
        #     "IsCallFlag	boolean,"
        #     "IsFtFAppointmentFlag	boolean,"
        #     "IsRPCFlag	boolean,"
        #     "IsRPCFlagCalc	boolean,"
        #     "IsGlobalRPCFlag	boolean,"
        #     "BaseCategory	string,"
        #     "FirstLevelCategory	string,"
        #     "SecondLevelCategory	string,"
        #     "ThirdLevelCategory	string,"
        #     "NotificationType	string,"
        #     "Hierarchy	int,"
        #     "InteractionDate	timestamp,"
        #     "TaskDate	timestamp,"
        #     "InteractionComments	string,"
        #     "ResultsComments string,"
        #     "DebtorStatus string,"
        #     "DebtorStatusComments string,"
        #     "CurrentStrategy string,"
        #     "CategoryOfAction string,"
        #     "SubCategoryOfAction string,"
        #     "Comments string,"
        #     "CaseCode	int,"
        #     "CaseCodePrimary	string,"
        #     "ContactPerson	string,"
        #     "ContactPersonCategoryCode	int,"
        #     "ContactPersonCategory	int,"
        #     "InteractionPhone	string,"
        #     "ExternalCompanyKey	int,"
        #     "CreatedDate	timestamp,"
        #     "CreatedBy	string,"
        #     "LastUpdatedDate	date,"
        #     "LastUpdatedBy	string,"
        #     "PromiseKey	int,"
        #     "PromiseAmount	decimal(19,4),"
        #     "PromiseTypeCode	int,"
        #     "PromiseType	int,"
        #     "PromiseToPayTypeCode	int,"
        #     "PromiseToPayType	string,"
        #     "PromiseDate	timestamp,"
        #     "PromiseDueDate	string,"
        #     "PromiseStatusCode	int,"
        #     "PromiseStatus	string,"
        #     "PromiseToPayAmount	decimal(19,4),"
        #     "PromiseComments	string,"
        #     "PaymentKey	int,"
        #     "PromiseLastUpdatedDate	timestamp,"
        #     "PromiseLastUpdatedBy	string,"
        #     "CoverageOfPaymentVSPromiseAmount	decimal(19,4),"
        #     "CoverageOfPromiseVSPaymentAmount	int,"
        #     "DaysBetweenPromiseAndDueDate	int,"
        #     "ScheduleDate date,"
        #     "BusinessDate date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Interaction"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "whl",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")
        # ================== debtor_status_action_participants ==================

        debtor_status_action_participants_extra_filters = None

        if debtor_status_action_participants_extra_filters is not None:
            debtor_status_action_participants_extra_filters.append(date_filter)
            debtor_status_action_participants_final_filters = " AND ".join(
                debtor_status_action_participants_extra_filters
            )
        else:
            debtor_status_action_participants_final_filters = date_filter

        debtor_status_action_participants_selected_fields = [
            "DebStatActParticipantsDescription",
            "DebStatActParticipantsActionId",
            "DebStatActParticipantsCategoryId",
            "DebStatActParticipantsId",
            "DebStatActParticipantsCdi",
        ]

        debtor_status_action_participants = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "whl_debtorstatusactionparticipants",
            debtor_status_action_participants_final_filters,
            debtor_status_action_participants_selected_fields,
        )
        # ================== debtor_status_action ==================

        debtor_status_action_extra_filters = None

        if debtor_status_action_extra_filters is not None:
            debtor_status_action_extra_filters.append(date_filter)
            debtor_status_action_final_filters = " AND ".join(
                debtor_status_action_extra_filters
            )
        else:
            debtor_status_action_final_filters = date_filter

        debtor_status_action_selected_fields = [
            "DebtorStatusActionDate",
            "DebtorStatusActionDueDate",
            "DebtorStatusActionID",
            "DebtorStatusActionSubCatID",
            "DebtorStatusActionComments",
            "debtorstatusactioncommunicationphonenumber",
            "DebtorStatusActionModifiedDate",
            "DebtorStatusActionModifiedByUser",
            "DebtorStatusID",
            "CategoryDescription",
            "SubCategoryDescription",
            "businessdate",
        ]

        debtor_status_action = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "whl_debtorstatusaction",
            debtor_status_action_final_filters,
            debtor_status_action_selected_fields,
        )
       
        debtor_status_action = (
            debtor_status_action
            .withColumnRenamed("SubCategoryDescription", "SubCategoryDescription_debtor_status_action")
            .withColumnRenamed("CategoryDescription", "CategoryDescription_debtor_status_action")
        )
        # ================== debtor_status ==================

        debtor_status_extra_filters = None

        if debtor_status_extra_filters is not None:
            debtor_status_extra_filters.append(date_filter)
            debtor_status_final_filters = " AND ".join(debtor_status_extra_filters)
        else:
            debtor_status_final_filters = date_filter

        debtor_status_selected_fields = ["DebtorStatusID",
                                         "DebtorStatusPrfCDI",
                                         "PortfolioID",
                                         "DebtorStatusDataStatusValue",
                                         "DebtorStatusNextActionValue",
                                         "DebtorStatusStatusComments"
                                         ]

        debtor_status = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "whl_debtorstatus",
            debtor_status_final_filters,
            debtor_status_selected_fields,
        )

        # ================== debtor_status_action_result ==================

        debtor_status_action_result_extra_filters = None

        if debtor_status_action_result_extra_filters is not None:
            debtor_status_action_result_extra_filters.append(date_filter)
            debtor_status_action_result_final_filters = " AND ".join(
                debtor_status_action_result_extra_filters
            )
        else:
            debtor_status_action_result_final_filters = date_filter

        debtor_status_action_result_selected_fields = [
            "DebtorStatusActionResultID",
            "DebtorStatusActionResultComments"
        ]

        debtor_status_action_result = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "whl_debtorstatusactionresult",
            debtor_status_action_result_final_filters,
            debtor_status_action_result_selected_fields,
        )

        # ================== activities ==================

        activities_extra_filters = None

        if activities_extra_filters is not None:
            activities_extra_filters.append(date_filter)
            activities_final_filters = " AND ".join(activities_extra_filters)
        else:
            activities_final_filters = date_filter

        activities_selected_fields = [
            "SubCategoryID",
            "SubCategoryCode",
            "SubCategoryDescription",
            "ActionTypeId",
            "ActionType",
            "ActionCatID",
            "CategoryDescription",
            "isactive",
            "iscall",
            "IsFtFAppointment",
            "IsRPC",
        ]

        activities = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "sme_activities",
            activities_final_filters,
            activities_selected_fields,
        )

        activities = (
            activities.withColumn(
                "IsFtFAppointment", col("IsFtFAppointment").cast("string")
            )
            .withColumn("IsRPC", col("IsRPC").cast("string"))
            .withColumn("iscall", col("iscall").cast("string"))
            .withColumnRenamed("SubCategoryDescription", "SubCategoryDescription_activities")
            .withColumnRenamed("CategoryDescription", "CategoryDescription_activities")

        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "portfolio_id_optiverse",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        portfolio = portfolio.withColumn(
            "portfolio_id_optiverse", col("portfolio_id_optiverse").cast("int")
        )
        # ================== interaction_whl_df ==================

        interaction_whl_primary_key_fields = [
            debtor_status_action_participants.DebStatActParticipantsId,
            lit("whl"),
        ]

        interaction_whl_df = (
            debtor_status_action.join(
                debtor_status_action_participants,
                (
                    debtor_status_action["DebtorStatusActionID"]
                    == debtor_status_action_participants[
                        "DebStatActParticipantsActionId"
                    ]
                ),
                "inner",
            )
            .join(
                activities,
                (
                    debtor_status_action["DebtorStatusActionSubCatID"]
                    == activities["SubCategoryID"]
                ),
                "inner",
            )
            .join(
                debtor_status,
                (
                    debtor_status_action["DebtorStatusID"]
                    == debtor_status["DebtorStatusID"]
                ),
                "inner",
            )
            .join(
                portfolio,
                (portfolio["portfolio_id_optiverse"] == debtor_status["PortfolioID"]),
                "left",
            )
            .join(
                 debtor_status_action_result,
                 debtor_status_action["DebtorStatusActionID"] == debtor_status_action_result["DebtorStatusActionResultID"],
                 "left",
    )
        )

        contact_type_condition = (
            when(
                coalesce(col("IsFtFAppointment"), lit("false")) == "true",
                "Face To Face",
            )
            .when(coalesce(col("IsRPC"), lit("false")) == "true", "Right Party Contact")
            .when(coalesce(col("iscall"), lit("false")) == "true", "Contact")
            .otherwise("No Contact")
        )

        interaction_whl_df = (
            interaction_whl_df.withColumn(
                "InteractionId",
                sha2(concat_ws(";", *interaction_whl_primary_key_fields), 256),
            )
            .withColumn("ContactTypeCode", contact_type_condition)
            .withColumn("ContactType", contact_type_condition)
        )

        # ================== Selection of Final Columns ==================

        interaction_whl_df = interaction_whl_df.select(
            interaction_whl_df.InteractionId.cast("string").alias("InteractionId"),
            lit("whl").alias("SourceSystem"),
            portfolio.portfolio_id_optiverse.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchID"),
            interaction_whl_df.DebStatActParticipantsId.cast("string").alias(
                "InteractionKey"
            ),
            interaction_whl_df.DebStatActParticipantsId.alias("InteractionCode"),
            lit(None).alias("ExposureKey"),
            lit(None).alias("PartyKey"),
            interaction_whl_df.DebtorStatusPrfCDI.cast("string").alias("CDI"),
            interaction_whl_df.SubCategoryID.cast("string").alias(
                "InteractionTypeCode"
            ),
            interaction_whl_df.SubCategoryCode.alias("InteractionTypeKey"),
            interaction_whl_df.SubCategoryDescription_activities.alias("InteractionType"),
            interaction_whl_df.ActionTypeId.alias("InteractionGroupTypeCode"),
            interaction_whl_df.ActionType.alias("InteractionGroupType"),
            interaction_whl_df.ActionCatID.alias("InteractionCategoryCode"),
            interaction_whl_df.CategoryDescription_activities.alias("InteractionCategory"),
            lit(None).alias("DialerTypeCode"),
            lit(None).alias("DialerType"),
            interaction_whl_df.ContactTypeCode,
            interaction_whl_df.ContactType,
            lit(None).alias("AttemptTypeCode"),
            lit(None).alias("AttemptType"),
            lit(None).alias("AttemptOutcomeCode"),
            lit(None).alias("AttemptOutcome"),
            lit(None).alias("AttemptSpecialMetricCode"),
            lit(None).alias("AttemptSpecialMetric"),
            lit(None).cast("boolean").alias("IsPositiveFlag"),
            lit("0").cast("boolean").alias("IsPromiseFlag"),
            interaction_whl_df.isactive.cast("boolean").alias("isactiveFlag"),
            lit("0").cast("boolean").alias("IsLegalFlag"),
            interaction_whl_df.iscall.cast("boolean").alias("IsCallFlag"),
            interaction_whl_df.IsFtFAppointment.cast("boolean").alias(
                "IsFtFAppointmentFlag"
            ),
            interaction_whl_df.IsRPC.cast("boolean").alias("IsRPCFlag"),
            lit(None).cast("boolean").alias("IsRPCFlagCalc"),
            lit(None).cast("boolean").alias("IsGlobalRPCFlag"),
            lit(None).alias("BaseCategory"),
            lit(None).alias("FirstLevelCategory"),
            lit(None).alias("SecondLevelCategory"),
            lit(None).alias("ThirdLevelCategory"),
            lit(None).alias("NotificationType"),
            lit(None).cast("int").alias("Hierarchy"),
            interaction_whl_df.DebtorStatusActionDate.alias("InteractionDate"),
            lit(None).cast("timestamp").alias("TaskDate"),
            interaction_whl_df.DebtorStatusActionComments.alias("InteractionComments"),
            interaction_whl_df.DebtorStatusActionResultComments.alias("ResultComments"),
            interaction_whl_df.DebtorStatusDataStatusValue.alias("DebtorStatus"),
            interaction_whl_df.DebtorStatusStatusComments.alias("DebtorStatusComments"),
            interaction_whl_df.DebtorStatusNextActionValue.alias("CurrentStrategy"),
            interaction_whl_df.CategoryDescription_debtor_status_action.alias("CategoryOfAction"),
            interaction_whl_df.SubCategoryDescription_debtor_status_action.alias("SubCategoryOfAction"),
            interaction_whl_df.DebtorStatusStatusComments.alias("Comments"),
            lit(None).alias("CaseCode"),
            lit(None).alias("CaseCodePrimary"),
            interaction_whl_df.DebStatActParticipantsDescription.alias("ContactPerson"),
            interaction_whl_df.DebStatActParticipantsCategoryId.alias(
                "ContactPersonCategoryCode"
            ),
            interaction_whl_df.DebStatActParticipantsCategoryId.alias(
                "ContactPersonCategory"
            ),
            interaction_whl_df.debtorstatusactioncommunicationphonenumber.alias(
                "InteractionPhone"
            ),
            lit(None).alias("ExternalCompanyKey"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("CreatedBy"),
            interaction_whl_df.DebtorStatusActionModifiedDate.cast("date").alias(
                "LastUpdatedDate"
            ),
            interaction_whl_df.DebtorStatusActionModifiedByUser.alias("LastUpdatedBy"),
            lit(None).alias("PromiseKey"),
            lit(None).alias("PromiseAmount"),
            lit(None).alias("PromiseTypeCode"),
            lit(None).alias("PromiseType"),
            lit(None).alias("PromiseToPayTypeCode"),
            lit(None).alias("PromiseToPayType"),
            lit(None).alias("PromiseDate"),
            lit(None).alias("PromiseDueDate"),
            lit(None).alias("PromiseStatusCode"),
            lit(None).alias("PromiseStatus"),
            lit(None).alias("PromiseToPayAmount"),
            lit(None).alias("PromiseComments"),
            lit(None).alias("PaymentKey"),
            lit(None).alias("PromiseLastUpdatedDate"),
            lit(None).alias("PromiseLastUpdatedBy"),
            lit(None).alias("CoverageOfPaymentVSPromiseAmount"),
            lit(None).alias("CoverageOfPromiseVSPaymentAmount"),
            lit(None).alias("DaysBetweenPromiseAndDueDate"),
            interaction_whl_df.DebtorStatusActionDueDate.cast("date").alias("ScheduleDate"),
            interaction_whl_df.businessdate.alias("BusinessDate"),
        )
        # ================== Append ==================

        interaction_whl_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_interaction_whl(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)